# 3a — Gradient Descent on Toy Problems

Brute force finds the minimum of the cost surface by checking every grid
cell. **Gradient descent** instead starts somewhere, measures which way is
downhill, takes a step, and repeats. This notebook builds the intuition on
problems small enough to draw, then applies the same idea to scan matching.

**Learning goals**

1. Understand the update rule $x \leftarrow x - \alpha \nabla f(x)$.
2. See how the learning rate $\alpha$ controls speed, oscillation and divergence.
3. See why an elongated bowl makes gradient descent zig-zag.
4. Replace the brute-force grid search with gradient descent on the scan-matching cost.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## A 1D landscape

Start with a function of one variable that has two minima — one shallow,
one deep. The gradient (here just the derivative) tells us the slope; we
step in the *opposite* direction to go downhill.

In [ ]:
def f(x):
    return x**4 - 4 * x**2 + x

def grad_f(x):
    return 4 * x**3 - 8 * x + 1

def gradient_descent_1d(x0, learning_rate, steps=40):
    path = [x0]
    x = x0
    for _ in range(steps):
        x = x - learning_rate * grad_f(x)
        if abs(x) > 10:          # give up once we've clearly diverged
            break
        path.append(x)
    return np.array(path)

xs = np.linspace(-2.5, 2.5, 400)
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, lr in zip(axes, [0.01, 0.05, 0.2]):
    path = gradient_descent_1d(x0=2.0, learning_rate=lr)
    ax.plot(xs, f(xs), "k-", lw=1)
    ax.plot(path, f(path), "o-", ms=4, color="tab:red")
    ax.plot(path[-1], f(path[-1]), "*", ms=14, color="tab:green")
    ax.set_title(f"learning rate = {lr}   ({len(path) - 1} steps)")
    ax.set_xlabel("x")
    ax.set_ylim(-8, 12)
axes[0].set_ylabel("f(x)")
plt.show()

Three behaviours, all controlled by the learning rate:

* **Too small** — the path creeps, and after 40 steps it still hasn't arrived.
* **About right** — it converges in a handful of steps.
* **Too large** — each step overshoots, and the iterate bounces around or flies off.

Also note that gradient descent only finds *a* minimum, not necessarily the
lowest one. Starting from `x = 2` it lands in the right-hand basin. Try a
negative starting point to reach the deeper one on the left.

## A 2D bowl, and the zig-zag

Scan matching has at least two unknowns, so now the gradient is a vector
and the picture is a contour map. This bowl is deliberately stretched: it
is ten times steeper in `y` than in `x`.

In [ ]:
def g(p):
    x, y = p
    return x**2 + 10 * y**2

def grad_g(p):
    x, y = p
    return np.array([2 * x, 20 * y])

def gradient_descent(grad, p0, learning_rate, steps=50, tol=1e-6):
    p = np.array(p0, float)
    path = [p.copy()]
    for _ in range(steps):
        step = learning_rate * grad(p)
        p = p - step
        path.append(p.copy())
        if np.linalg.norm(step) < tol:
            break
    return np.array(path)

X, Y = np.meshgrid(np.linspace(-3, 3, 200), np.linspace(-1.5, 1.5, 200))
Z = X**2 + 10 * Y**2

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, lr in zip(axes, [0.02, 0.09, 0.105]):
    path = gradient_descent(grad_g, p0=(-2.5, 1.0), learning_rate=lr)
    ax.contour(X, Y, Z, levels=20, cmap="viridis")
    ax.plot(path[:, 0], path[:, 1], "o-", ms=3, color="tab:red")
    ax.set_title(f"learning rate = {lr}  ({len(path) - 1} steps)")
    ax.set_xlim(-3, 3); ax.set_ylim(-1.5, 1.5)
plt.show()

The steep `y` direction forces a small learning rate, which in turn makes
progress along the shallow `x` direction painfully slow — and pushing the
rate up makes `y` oscillate and finally diverge. Ill-conditioned bowls like
this are the motivation for the Gauss–Newton method in the next script,
which uses curvature information to take a well-scaled step in every direction.

## Where does the gradient come from?

We wrote `grad_f` and `grad_g` by hand. When that's inconvenient, a
**finite-difference** approximation works for any cost function:

$$\frac{\partial f}{\partial x_i} \approx \frac{f(x + h e_i) - f(x - h e_i)}{2h}.$$

In [ ]:
def numerical_gradient(func, p, h=1e-5):
    p = np.asarray(p, float)
    grad = np.zeros_like(p)
    for i in range(len(p)):
        e = np.zeros_like(p); e[i] = h
        grad[i] = (func(p + e) - func(p - e)) / (2 * h)
    return grad

p = np.array([1.0, -0.5])
print("analytic: ", grad_g(p))
print("numerical:", numerical_gradient(g, p))

## Gradient descent for scan matching

Back to the problem we care about. Using the corner scans from
[2b](2b_cost_function_surface.ipynb), the mean squared error over a
translation `(dx, dy)` is

$$\mathrm{MSE}(\mathbf{t}) = \frac{1}{N}\sum_i \|\mathbf{p}_i^{(0)} - \mathbf{p}_i^{(1)} - \mathbf{t}\|^2,
\qquad
\nabla \mathrm{MSE} = -\frac{2}{N}\sum_i \big(\mathbf{p}_i^{(0)} - \mathbf{p}_i^{(1)} - \mathbf{t}\big).$$

Using the *mean* rather than the *sum* keeps the gradient the same size no
matter how many points are in the scan, so one learning rate works for any
scan.

In [ ]:
def make_corner(num_points=40, length=4.0):
    s = np.linspace(0, length, num_points)
    return np.vstack((np.column_stack((s, np.zeros_like(s))),
                      np.column_stack((np.zeros_like(s), s))))

true_t = np.array([0.6, 0.3])
scan_t0 = make_corner()
scan_t1 = scan_t0 - true_t                 # robot moved by +true_t, scene shifted by -true_t

def match_mse(t):
    return np.mean((scan_t0 - (scan_t1 + t)) ** 2) * 2   # *2: sum over x and y, mean over points

def match_grad(t):
    return -2 * np.mean(scan_t0 - scan_t1 - t, axis=0)

path = gradient_descent(match_grad, p0=(0.0, 0.0), learning_rate=0.3, steps=30)
print("numerical gradient agrees:", np.allclose(match_grad(path[0]), numerical_gradient(match_mse, path[0])))
print(f"estimate after {len(path) - 1} steps: dx={path[-1, 0]:.3f}, dy={path[-1, 1]:.3f}  (true {true_t[0]}, {true_t[1]})")

In [ ]:
cand = np.arange(-0.5, 1.5, 0.05)
DX, DY = np.meshgrid(cand, cand)
C = np.array([[match_mse(np.array([dx, dy])) for dx in cand] for dy in cand])

fig, (ax_path, ax_conv) = plt.subplots(1, 2, figsize=(12, 5))
ax_path.contour(DX, DY, C, levels=25, cmap="viridis")
ax_path.plot(path[:, 0], path[:, 1], "o-", ms=4, color="tab:red", label="gradient descent")
ax_path.plot(*true_t, "r*", ms=14, label="true motion")
ax_path.set_xlabel("dx"); ax_path.set_ylabel("dy"); ax_path.axis("equal"); ax_path.legend()
ax_path.set_title("Walking down the scan-matching bowl")

ax_conv.semilogy([match_mse(p) for p in path], "o-", ms=4)
ax_conv.set_xlabel("iteration"); ax_conv.set_ylabel("MSE"); ax_conv.grid(True)
ax_conv.set_title("Cost per iteration")
plt.show()

The brute-force search in step 1 evaluated 441 candidate translations to
reach the same answer on a 0.1 m grid. Gradient descent got there in a
dozen evaluations *and* is not limited to grid resolution.

## Summary

* Gradient descent: measure the slope, step downhill, repeat.
* The learning rate trades speed for stability; elongated bowls make the
  trade-off painful.
* Finite differences give a gradient for any cost you can evaluate.
* On the scan-matching cost, gradient descent replaces a grid search with a
  short walk to the minimum.

**Next:** the Gauss–Newton script (`3b`) fixes the zig-zag by using
curvature, and Module 4 removes the assumption that we know which point
matches which — starting with [4a — Nearest neighbours](4a_nearest_neighbors_kdtree.ipynb).